In [1]:
import sqlite3

DB_PATH = r"C:\Users\nikol\Desktop\disser\newspapers.db"

def create_database():
    """Создаёт базу данных и таблицу newspaper_mentions."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS newspaper_mentions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            newspaper_name TEXT NOT NULL,
            publication_date TEXT NOT NULL,
            person_name TEXT,
            context TEXT,
            pdf_filename TEXT
        )
    """)

    conn.commit()
    conn.close()
    print(f"База данных '{DB_PATH}' и таблица 'newspaper_mentions' созданы.")

if __name__ == "__main__":
    create_database()

База данных 'C:\Users\nikol\Desktop\disser\newspapers.db' и таблица 'newspaper_mentions' созданы.


In [ ]:
import sqlite3
import pandas as pd

DB_PATH = "newspapers.db"
OUTPUT_XLSX = "newspaper_mentions.xlsx"


def export_mentions_to_excel(
    db_path: str = DB_PATH,
    out_xlsx: str = OUTPUT_XLSX
) -> None:
    """
    Выгружает таблицу newspaper_mentions из SQLite в XLSX.
    """
    conn = sqlite3.connect(db_path)

    # Можно выбрать только нужные поля / сделать сортировку
    query = """
        SELECT
            newspaper_name       AS Газета,
            publication_date     AS Дата_выпуска,
            person_name          AS Имя,
            context              AS Контекст,
            pdf_filename         AS PDF_файл
        FROM newspaper_mentions
        ORDER BY publication_date, newspaper_name, person_name
    """

    df = pd.read_sql_query(query, conn)
    conn.close()

    # Запись в Excel
    with pd.ExcelWriter(out_xlsx, engine="openpyxl") as writer:
        df.to_excel(writer, sheet_name="mentions", index=False)

    print(f"Данные успешно выгружены в '{out_xlsx}'.")


if __name__ == "__main__":
    export_mentions_to_excel()

Данные успешно выгружены в 'newspaper_mentions.xlsx'.


In [2]:
import sqlite3
import pandas as pd

DB_PATH = r"C:\Users\nikol\Desktop\disser\newspapers.db"
EXCEL_PATH = r"C:\Users\nikol\Desktop\disser\newspaper_mentions.xlsx"  # путь к вашему xlsx


def create_database():
    """Создаёт базу данных и таблицу newspaper_mentions, если её ещё нет."""
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()

    cursor.execute("""
        CREATE TABLE IF NOT EXISTS newspaper_mentions (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            newspaper_name TEXT NOT NULL,
            publication_date TEXT NOT NULL,
            person_name TEXT,
            context TEXT,
            pdf_filename TEXT
        )
    """)

    conn.commit()
    conn.close()
    print("Таблица newspaper_mentions проверена/создана.")


def load_from_excel():
    """Читает данные из Excel и загружает в SQLite."""
    # прочитать первый лист
    df = pd.read_excel(EXCEL_PATH)

    # если у вас в xlsx есть колонка 'id', она вам в БД не нужна (id автоинкрементное)
    # поэтому удаляем её, если она присутствует
    if 'id' in df.columns:
        df = df.drop(columns=['id'])

    # убедимся, что названия столбцов совпадают с полями таблицы
    # ожидаются: newspaper_name, publication_date, person_name, context, pdf_filename
    print("Найденные столбцы в Excel:", df.columns.tolist())

    # при необходимости можно переименовать столбцы, если в xlsx они отличаются
    # пример:
    # df = df.rename(columns={
    #     'название_газеты': 'newspaper_name',
    #     'дата': 'publication_date',
    #     'персона': 'person_name',
    #     'контекст': 'context',
    #     'файл': 'pdf_filename',
    # })

    # преобразуем дату к строке в формате YYYY-MM-DD (если это настоящий тип даты)
    if 'publication_date' in df.columns:
        df['publication_date'] = pd.to_datetime(df['publication_date']).dt.strftime('%Y-%m-%d')

    # подставим None вместо NaN, чтобы корректно писалось в SQLite
    df = df.where(pd.notnull(df), None)

    conn = sqlite3.connect(DB_PATH)

    # вставка в таблицу, без замены, просто добавляем строки
    df.to_sql('newspaper_mentions', conn, if_exists='append', index=False)

    conn.close()
    print(f"Загружено строк: {len(df)}")


if __name__ == "__main__":
    create_database()
    load_from_excel()

Таблица newspaper_mentions проверена/создана.
Найденные столбцы в Excel: ['newspaper_name', 'publication_date', 'person_name', 'context']
Загружено строк: 1356


In [4]:
import sqlite3

DB_PATH = r"C:\Users\nikol\Desktop\disser\newspapers.db"

conn = sqlite3.connect(DB_PATH)
cur = conn.cursor()

# 1. Переименовать старую таблицу
cur.execute("ALTER TABLE newspaper_mentions RENAME TO newspaper_mentions_old;")

# 2. Создать новую с теми же полями, но без NOT NULL на newspaper_name
cur.execute("""
    CREATE TABLE newspaper_mentions (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        newspaper_name TEXT,
        publication_date TEXT NOT NULL,
        person_name TEXT,
        context TEXT,
        pdf_filename TEXT
    );
""")

# 3. Перенести данные
cur.execute("""
    INSERT INTO newspaper_mentions (id, newspaper_name, publication_date, person_name, context, pdf_filename)
    SELECT id, newspaper_name, publication_date, person_name, context, pdf_filename
    FROM newspaper_mentions_old;
""")

# 4. Удалить старую таблицу
cur.execute("DROP TABLE newspaper_mentions_old;")

conn.commit()
conn.close()

print("Структура таблицы newspaper_mentions обновлена: newspaper_name теперь может быть NULL.")

Структура таблицы newspaper_mentions обновлена: newspaper_name теперь может быть NULL.
